In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t072/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "inventory接口流量下跌告警" (inventory interface traffic drop alert). It occurred for the operation `inventory::/api/v1/inventory/{productId}` in the APM domain. The alert time window is from `2026-04-04 22:37:09.273091` to `2026-04-04 22:41:54` (UTC+8). The user is requesting help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('', {'start': '2026-04-04T22:37:09.273091+08:00', 'end': '2026-04-04T22:41:54+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">inventory接口流量下跌告警<alert_event event_id="eb1bb24cd37c2c0849adec1e511db312" trans_id="nkucnv79s905l7f5b3t8na9c06" rule_id="b7853e06-54b7-4409-9eea-9bb82fbfb252" rule_name="inventory接口流量下跌告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t072/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Ensure 'time' and 'value' columns are numeric
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Aggregate KPIs for all components by 'component-KPI'
metrics_data['component_KPI'] = metrics_data['entity_name'] + '-' + metrics_data['metric']
aggregated_kpis = metrics_data.groupby(['component_KPI', 'time'])['value'].sum().reset_index()

# Calculate global thresholds (e.g., P95) for each 'component-KPI'
global_thresholds = aggregated_kpis.groupby('component_KPI')['value'].quantile(0.95).reset_index()
global_thresholds.columns = ['component_KPI', 'P95']

aggregated_kpis, global_thresholds

```
Out[2]:
```
The `metrics.parquet` file was successfully loaded, and KPIs were aggregated for all components, resulting in 31,809 time series data points classified by 'component-KPI'. Global thresholds (P95) were calculated for each 'component-KPI', producing 298 unique thresholds. These thresholds represent the 95th percentile of metric values for each component-KPI across the entire dataset.

The original code execution output of IPython Kernel is also provided below for reference:

(                                           component_KPI              time   value
0                      -apiserver_concurrent_connections  1775309829000000     4.0
1                      -apiserver_concurrent_connections  1775309859000000     4.0
2                      -apiserver_concurrent_connections  1775309889000000     4.0
3                      -apiserver_concurrent_connections  1775309919000000     5.0
4                      -apiserver_concurrent_connections  1775309949000000     4.0
...                                                  ...               ...     ...
31804  shipping::oteldemo.ShippingService/ShipOrder-r...  1775313699000000  1454.0
31805  shipping::oteldemo.ShippingService/ShipOrder-r...  1775313729000000  2224.0
31806  shipping::oteldemo.ShippingService/ShipOrder-r...  1775313759000000  1846.0
31807  shipping::oteldemo.ShippingService/ShipOrder-r...  1775313789000000  1515.0
31808  shipping::oteldemo.ShippingService/ShipOrder-r...  1775313819000000  1235.0

[31809 rows x 3 columns],                                          component_KPI           P95
0                    -apiserver_concurrent_connections      5.000000
1                  -apiserver_non_2xx_read_request_qps      0.088371
2                 -apiserver_non_2xx_write_request_qps      0.088282
3                       -apiserver_requests_per_second     26.439192
4                           -cluster_failed_node_count      0.000000
..                                                 ...           ...
293                                  shipping-workload  12393.250000
294  shipping::oteldemo.ShippingService/GetQuote-av...      0.002894
295  shipping::oteldemo.ShippingService/GetQuote-re...   6199.650000
296  shipping::oteldemo.ShippingService/ShipOrder-a...      0.000009
297  shipping::oteldemo.ShippingService/ShipOrder-r...   6192.550000

[298 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

from datetime import datetime
import pytz

# Convert alert window timestamps to Unix microseconds for filtering
timezone = pytz.timezone('Asia/Shanghai')
alert_start = int(datetime.strptime(alert_window['start'], '%Y-%m-%dT%H:%M:%S.%f%z').timestamp() * 1_000_000)
alert_end = int(datetime.strptime(alert_window['end'], '%Y-%m-%dT%H:%M:%S%z').timestamp() * 1_000_000)

# Filter the aggregated 'component-KPI' time series data within the alert time window
filtered_kpis = aggregated_kpis[(aggregated_kpis['time'] >= alert_start) & (aggregated_kpis['time'] <= alert_end)]

# Merge filtered data with global thresholds to compare values against P95
anomalies = filtered_kpis.merge(global_thresholds, on='component_KPI')
anomalies['is_anomalous'] = anomalies['value'] > anomalies['P95']

# Filter anomalies where values exceed the P95 threshold
anomalous_kpis = anomalies[anomalies['is_anomalous']]

filtered_kpis, anomalous_kpis

```
Out[3]:
```
Within the alert time window, 2,354 data points were filtered from the aggregated 'component-KPI' time series. Among these, 516 anomalies were identified where the metric values exceeded their respective P95 global thresholds. These anomalies indicate potential issues in specific components or KPIs during the alert period.

The original code execution output of IPython Kernel is also provided below for reference:

(                                           component_KPI              time   value
121                    -apiserver_concurrent_connections  1775313459000000     4.0
122                    -apiserver_concurrent_connections  1775313489000000     7.0
123                    -apiserver_concurrent_connections  1775313519000000     4.0
124                    -apiserver_concurrent_connections  1775313549000000     7.0
125                    -apiserver_concurrent_connections  1775313579000000     4.0
...                                                  ...               ...     ...
31800  shipping::oteldemo.ShippingService/ShipOrder-r...  1775313579000000  1354.0
31801  shipping::oteldemo.ShippingService/ShipOrder-r...  1775313609000000  1491.0
31802  shipping::oteldemo.ShippingService/ShipOrder-r...  1775313639000000  2299.0
31803  shipping::oteldemo.ShippingService/ShipOrder-r...  1775313669000000  1517.0
31804  shipping::oteldemo.ShippingService/ShipOrder-r...  1775313699000000  1454.0

[2354 rows x 3 columns],                                           component_KPI              time         value           P95  is_anomalous
1                     -apiserver_concurrent_connections  1775313489000000  7.000000e+00  5.000000e+00          True
3                     -apiserver_concurrent_connections  1775313549000000  7.000000e+00  5.000000e+00          True
12                  -apiserver_non_2xx_read_request_qps  1775313549000000  8.906306e-02  8.837084e-02          True
19                 -apiserver_non_2xx_write_request_qps  1775313489000000  2.656129e-01  8.828153e-02          True
20                 -apiserver_non_2xx_write_request_qps  1775313519000000  8.829554e-01  8.828153e-02          True
...                                                 ...               ...           ...           ...           ...
2294                                       shipping-mem  1775313549000000  3.275435e+06  3.261918e+06          True
2339  shipping::oteldemo.ShippingService/ShipOrder-a...  1775313549000000  9.110185e-06  8.995567e-06          True
2340  shipping::oteldemo.ShippingService/ShipOrder-a...  1775313579000000  9.447984e-06  8.995567e-06          True
2341  shipping::oteldemo.ShippingService/ShipOrder-a...  1775313609000000  9.398725e-06  8.995567e-06          True
2344  shipping::oteldemo.ShippingService/ShipOrder-a...  1775313699000000  9.430448e-06  8.995567e-06          True

[516 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Group consecutive anomalies for each 'component-KPI' to locate faults
anomalous_kpis['time_diff'] = anomalous_kpis.groupby('component_KPI')['time'].diff().fillna(0)
anomalous_kpis['is_consecutive'] = anomalous_kpis['time_diff'] <= 300_000_000  # 5 minutes in microseconds

# Calculate the extremal breach for each 'component-KPI'
anomalous_kpis['breach_ratio'] = (anomalous_kpis['value'] - anomalous_kpis['P95']) / anomalous_kpis['P95']

# Exclude isolated noise spikes where the threshold breach is minimal (<= 50% of the extremal)
faults = anomalous_kpis[
    (anomalous_kpis['is_consecutive']) & (anomalous_kpis['breach_ratio'] > 0.5)
]

faults.reset_index(drop=True, inplace=True)
faults

```
Out[4]:
```
The analysis identified 104 faults where consecutive anomalies exceeded the P95 threshold by more than 50% of the extremal breach. These faults are grouped by 'component-KPI' and represent significant deviations from normal behavior, excluding isolated noise spikes. Key components and KPIs, such as `-apiserver_non_2xx_write_request_qps`, `ad-865bb86b84-6hp49-avg_request_latency_seconds`, and `inventory::/api/v1/inventory/{productId}-avg_request_latency_seconds`, show substantial breaches, indicating potential root causes for the alert.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI              time       value        P95  is_anomalous   time_diff  is_consecutive  breach_ratio
0                 -apiserver_non_2xx_write_request_qps  1775313489000000    0.265613   0.088282          True         0.0            True      2.008703
1                 -apiserver_non_2xx_write_request_qps  1775313519000000    0.882955   0.088282          True  30000000.0            True      9.001587
2      ad-865bb86b84-6hp49-avg_request_latency_seconds  1775313549000000    0.027677   0.013976          True         0.0            True      0.980357
3      ad-865bb86b84-6hp49-avg_request_latency_seconds  1775313579000000    0.024020   0.013976          True  30000000.0            True      0.718663
4      ad-865bb86b84-6hp49-avg_request_latency_seconds  1775313639000000    0.044710   0.013976          True  60000000.0            True      2.199076
..                                                 ...               ...         ...        ...           ...         ...             ...           ...
99   inventory::/actuator/health/**-avg_request_lat...  1775313699000000    1.749636   0.814329          True  30000000.0            True      1.148562
100  inventory::/api/v1/inventory/{productId}-avg_r...  1775313639000000    9.944133   3.682673          True  30000000.0            True      1.700249
101       recommendation-95bcd75df-rxwhh-request_count  1775313519000000  105.000000  56.450000          True  30000000.0            True      0.860053
102       recommendation-95bcd75df-rxwhh-request_count  1775313669000000   99.000000  56.450000          True  60000000.0            True      0.753764
103       recommendation-95bcd75df-rxwhh-request_count  1775313699000000   97.000000  56.450000          True  30000000.0            True      0.718335

[104 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t072/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure 'startTime' and 'endTime' are numeric
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Filter traces within the alert time window
filtered_traces = traces_data[
    (traces_data['startTime'] / 1_000_000 >= alert_start) &
    (traces_data['endTime'] / 1_000_000 <= alert_end)
]

# Focus on services related to the identified faults
faulty_services = faults['component_KPI'].str.split('-').str[0].unique()
related_traces = filtered_traces[filtered_traces['serviceName'].isin(faulty_services)]

# Identify the most downstream faulty service in the call chain
# A downstream service will have no other services calling it (no parent spans)
related_traces['is_downstream'] = ~related_traces['spanId'].isin(related_traces['parentSpanId'])
downstream_faulty_services = related_traces[related_traces['is_downstream']]

downstream_faulty_services[['traceId', 'spanId', 'serviceName', 'spanName', 'startTime', 'endTime']]

```
Out[5]:
```
No downstream faulty services were identified in the call chain within the alert time window. This indicates that either the faults are not directly related to downstream services or the traces do not provide sufficient evidence to pinpoint a specific downstream service as the root cause. Further investigation may be needed to analyze upstream dependencies or other telemetry data.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanId, serviceName, spanName, startTime, endTime]
Index: []```
```

In [ ]:
# In[6]:

# Revise the code to exclude the '_node_name_' column, which is not present in the logs data
# Load the logs.parquet file
logs_file_path = 'rca100/cases/t072/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is numeric
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs within the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start) & (logs_data['_time_'] <= alert_end)
]

# Search for error or warning messages related to the 'inventory' service
inventory_logs = filtered_logs[
    filtered_logs['content'].str.contains('inventory', case=False, na=False) &
    filtered_logs['content'].str.contains('error|warn', case=False, na=False)
]

inventory_logs[['content', '_time_', '_pod_name_']]

```
Out[6]:
```
No error or warning messages related to the `inventory` service were found in the logs within the alert time window. This suggests that the root cause of the issue may not be directly reflected in the application logs for the `inventory` service. Further analysis of other telemetry data or upstream/downstream dependencies may be required to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _pod_name_]
Index: []```
```

In [ ]:
# In[7]:

# Load the topology.json file
topology_file_path = 'rca100/cases/t072/topology.json'

with open(topology_file_path, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges from the topology data
entities = pd.DataFrame(topology_data['entities'])
edges = pd.DataFrame(topology_data['edges'])

# Focus on the `inventory` service and its operation `/api/v1/inventory/{productId}`
inventory_service = 'inventory'
inventory_operation = '/api/v1/inventory/{productId}'

# Identify the `inventory` service entity
inventory_entity = entities[
    (entities['name'].str.contains(inventory_service, case=False, na=False)) &
    (entities['name'].str.contains(inventory_operation, case=False, na=False))
]

# Trace upstream and downstream components related to the `inventory` service
if not inventory_entity.empty:
    inventory_entity_id = inventory_entity.iloc[0]['id']
    upstream_components = edges[edges['dst'] == inventory_entity_id]
    downstream_components = edges[edges['src'] == inventory_entity_id]
else:
    upstream_components = pd.DataFrame()
    downstream_components = pd.DataFrame()

inventory_entity, upstream_components, downstream_components

```
Out[7]:
```
The `inventory` operation `/api/v1/inventory/{productId}` was identified in the service dependency graph. It is contained within the `inventory` service, but no downstream components were found that depend on this operation. This indicates that the `inventory` operation is likely at the end of the call chain, and no fault propagation paths to other components were detected.

The original code execution output of IPython Kernel is also provided below for reference:

(                                   id           type                                      name  first_observed  last_observed                                              props
113  162799adab6799326676266a2c96cf3e  apm.operation  inventory::/api/v1/inventory/{productId}               0     1775317453  {'service': 'inventory', 'operation': '/api/v1...,                                  src     src_type                               dst       dst_type  relation  first_observed  last_observed
12  ed13cb14265f98d9eb3b89b664760adb  apm.service  162799adab6799326676266a2c96cf3e  apm.operation  contains               0     1775234653, Empty DataFrame
Columns: [src, src_type, dst, dst_type, relation, first_observed, last_observed]
Index: [])```
```